In [24]:
import numpy as np
import cv2
import os
import math
from pathlib import Path
import matplotlib.pyplot as plt
import re

def rotate_image(image, angle_degrees):
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return rotated

def numerical_sort_key(path):
    numbers = re.findall(r'\d+', path.stem)
    return int(numbers[0]) if numbers else 0

def crop_top_center_square(image, center, radius):
    h, w = image.shape[:2]
    side = int(radius)
    x1 = max(center[0] - side // 2, 0)
    x2 = min(center[0] + side // 2, w)
    y1 = 0
    y2 = min(side, h)
    return image[y1:y2, x1:x2]

def process_and_save_data_across_folders(
    img_parent_dir, 
    label_parent_dir,
    output_dir, 
    forces_name, 
    positions_name, 
    angles_name, 
    test_mode=False,
    save=0
):
    img_parent_dir = Path(img_parent_dir)
    label_parent_dir = Path(label_parent_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all subfolders named by integers, sorted numerically
    img_subfolders = sorted([f for f in img_parent_dir.iterdir() if f.is_dir() and f.name.isdigit()], key=lambda x: int(x.name))
    label_subfolders = sorted([f for f in label_parent_dir.iterdir() if f.is_dir() and f.name.isdigit()], key=lambda x: int(x.name))
    
    count = 0
    label_list = []
    
    for img_folder, label_folder in zip(img_subfolders, label_subfolders):
        forces_path = label_folder / forces_name
        positions_path = label_folder / positions_name
        angles_path = label_folder / angles_name
        if not (forces_path.exists() and positions_path.exists() and angles_path.exists()):
            print(f"Skipping {label_folder}, missing required files.")
            continue
        forces = np.load(forces_path)
        contact_angles = np.load(positions_path)
        force_angles = np.load(angles_path)
        image_files = sorted([f for f in img_folder.iterdir() if f.suffix.lower() in ['.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff']], key=numerical_sort_key)
        if test_mode:
            image_files = image_files[:2250]
            forces = forces[:2250]
            contact_angles = contact_angles[:2250]
            force_angles = force_angles[:2250]
        images = []
        for img_file in image_files:
            img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"Warning: Could not read {img_file}")
                continue
            images.append(img)
        if not images:
            continue
        images = np.stack(images, axis=0)
        num_images = images.shape[0]
        h, w = images.shape[1], images.shape[2]
        center_x, center_y = w // 2, h // 2
        radius = min(center_x, center_y)
        for i in range(num_images):
            img = images[i]
            if img.dtype != np.uint8:
                if img.max() <= 1.0: 
                    img = (img * 255).astype(np.uint8)
                else:
                    img = img.astype(np.uint8)
            num_contacts = forces.shape[1]
            for j in range(num_contacts):
                force_mag = forces[i, j]
                force_ang = force_angles[i, j]
                contact_angle_deg = contact_angles[i, j]/np.pi*180
                if force_mag == 0:
                    continue
                rotation_needed = 90+contact_angle_deg
                rotated_img = rotate_image(img, rotation_needed)
                cropped_img = crop_top_center_square(rotated_img, (center_x, center_y), radius)
                img_index = count + 1
                filename = f"{img_index:05d}.png"
                save_path = output_dir / filename
                if save:
                    cv2.imwrite(str(save_path), cropped_img)
                label_list.append([force_mag, force_ang])
                count += 1
                if 0:
                    plt.imshow(cropped_img, cmap='gray')
                    plt.title(f"Image {img_index:05d}, Force {force_mag:.4f}, Angle {force_ang:.4f}")
                    plt.axis('off')
                    plt.show()
    label_array = np.array(label_list)
    if save:
        np.save(output_dir / "labels.npy", label_array)
    print(f"Done! Saved {count} cropped contact images to {output_dir}")

In [25]:
process_and_save_data_across_folders(
        img_parent_dir='C:\\Users\\jcTSAI\\Desktop\\PE_Force_ML_Data\\synthImage_poly_stress_202511\\test\\',
        label_parent_dir='C:\\Users\\jcTSAI\\Desktop\\PE_Force_ML_Data\\synth_labels_poly_stress_202511\\test\\',
        output_dir='C:\\Users\\jcTSAI\\Desktop\\PE_Force_ML_Data\\synth_img_poly_stress_contact_crop_202601',
        forces_name='mags_test.npy',
        positions_name='angles_inner_test.npy',
        angles_name='angles_tang_test.npy',
        test_mode=1, save=1
    )

Done! Saved 31500 cropped contact images to C:\Users\jcTSAI\Desktop\PE_Force_ML_Data\synth_img_poly_stress_contact_crop_202601
